# Tester Notebook — Clean Qwen Base Model

Load a clean, read-only copy of the base model used by `09_original.ipynb` and separately display that experiment's already-saved metrics. This notebook does **not** train, evaluate, or run inference.

In [ ]:
from pathlib import Path
import gc
import json
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

## Locate the saved `09_original` metrics

In [ ]:
def find_project_root(start=Path.cwd()):
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "code" / "final_submission").is_dir():
            return candidate
    runpod_checkout = Path("/workspace/qub-machine-unlearning")
    if (runpod_checkout / "code" / "final_submission").is_dir():
        return runpod_checkout
    raise FileNotFoundError("Could not locate the project root.")

PROJECT_ROOT = find_project_root()
FINAL_SUBMISSION = PROJECT_ROOT / "code" / "final_submission"
BASELINE_RUN_ID = "20260829T151430Z"
RUNPOD_OUTPUT_ROOT = Path("/workspace/qwen35_classifier_runpod")

In [ ]:
RUNPOD_RESULTS = RUNPOD_OUTPUT_ROOT / "results" / BASELINE_RUN_ID
LOCAL_RESULTS = FINAL_SUBMISSION / "results" / "qwen" / "original"
if (RUNPOD_RESULTS / "test_metrics.csv").is_file():
    METRICS_PATH = RUNPOD_RESULTS / "test_metrics.csv"
else:
    METRICS_PATH = LOCAL_RESULTS / "retained_test_metrics.csv"

MODEL_ID = "unsloth/Qwen3.5-2B-Base"
MAX_SEQ_LENGTH = 512
print("Clean base checkpoint:", MODEL_ID)
print("Saved metrics:", METRICS_PATH)

## View the saved baseline metrics

These values are read from the final CSV produced by `09_original`; the model is not evaluated again.

In [ ]:
if not METRICS_PATH.is_file():
    raise FileNotFoundError(f"Saved baseline metrics not found: {METRICS_PATH}")

baseline_metrics = pd.read_csv(METRICS_PATH)
if len(baseline_metrics) != 1:
    raise ValueError(f"Expected one metrics row, found {len(baseline_metrics)}.")

print(f"Loaded saved metrics from:\n  {METRICS_PATH}")
display(baseline_metrics.T.rename(columns={0: "Original Qwen"}))

## Load a clean base-model copy

This loads the untouched Hugging Face base checkpoint used at the start of `09_original.ipynb`. It deliberately does not attach the trained LoRA adapter or classification head. Loading only: no prediction, metric calculation, optimiser, or training step is performed.

In [ ]:
# Loading a multi-billion-parameter model is opt-in so opening or running
# the metrics cells cannot allocate GPU memory accidentally.
LOAD_CLEAN_BASE_MODEL = False

In [ ]:
def load_clean_qwen_base():
    from unsloth import FastVisionModel

    model, processor = FastVisionModel.from_pretrained(
        MODEL_ID, load_in_4bit=False, load_in_16bit=True,
        max_seq_length=MAX_SEQ_LENGTH, use_gradient_checkpointing=False,
    )
    tokenizer = getattr(processor, "tokenizer", processor)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.requires_grad_(False)
    model.eval()
    return model, processor, tokenizer

In [ ]:
if LOAD_CLEAN_BASE_MODEL:
    if not torch.cuda.is_available():
        raise RuntimeError("The exact 09_original loading path requires a CUDA runtime with Unsloth.")
    model, processor, tokenizer = load_clean_qwen_base()
    device = next(model.parameters()).device
    print(f"Loaded clean frozen base model: {MODEL_ID}")
    print(f"Device: {device}")
    print(f"Training enabled: {any(p.requires_grad for p in model.parameters())}")
    print("No inference or training has been run.")
else:
    model = processor = tokenizer = None
    print("Clean base-model loading is disabled. Set LOAD_CLEAN_BASE_MODEL = True on the CUDA/Unsloth runtime to load it.")